In [1]:
import sys
print("Python version:", sys.version)
print("Setup verified ")

Python version: 3.11.1 (tags/v3.11.1:a7a450f, Dec  6 2022, 19:58:39) [MSC v.1934 64 bit (AMD64)]
Setup verified 


In [2]:
# Dependencies managed via requirements.txt
# pip install pyspark pandas python-dotenv

In [3]:
import os
from pathlib import Path 

def _find_project_root(marker: str = "pyproject.toml") -> Path:
    current = Path(os.path.abspath("")). resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate 
    return current.parent

PROJECT_ROOT = _find_project_root()
BRONZE_BASE = str(PROJECT_ROOT / "data" / "bronze" / "stocks")

partitions = [d for d in os.listdir(BRONZE_BASE) if d.startswith("date=")]
latest = sorted(partitions)[-1]
partition_dir = os.path.join(BRONZE_BASE, latest)

# Robust check: try stocks_raw.csv, then any csv
target_file = os.path.join(partition_dir, "stocks_raw.csv")
if not os.path.exists(target_file):
    csv_files = sorted([f for f in os.listdir(partition_dir) if f.endswith(".csv")])
    if csv_files:
        target_file = os.path.join(partition_dir, csv_files[-1])

BRONZE_FILE = target_file

if os.path.exists(BRONZE_FILE):
    size_mb = os.path.getsize(BRONZE_FILE) / (1024 * 1024)
    print(f" Bronze stocks file found: {BRONZE_FILE}")
    print(f" Size: {size_mb:.2f} MB")
    print(f"   Partition: {latest}")
else:
    raise FileNotFoundError(f" Bronze stocks file not found in: {partition_dir}")

print(f"\n Bronze stocks path configured")


 Bronze stocks file found: C:\FinPulse Project\data\bronze\stocks\date=2026-03-29\stocks_raw_185807.csv
 Size: 0.37 MB
   Partition: date=2026-03-29

 Bronze stocks path configured


In [4]:
# ============================================================
# FinPulse — Silver Layer: Stock Market Data
# Purpose: Clean, enrich and calculate metrics for stocks
# Storage: Apache Iceberg with Hadoop Catalog
# Stability: Multi-layer Win10/8GB Fixes applied
# ============================================================

import os 
import sys
from pathlib import Path
from datetime import datetime 
from dotenv import load_dotenv

load_dotenv(override=True)

def _find_project_root(marker: str = "pyproject.toml") -> Path:
    current = Path(os.getcwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate 
    return current

PROJECT_ROOT = _find_project_root()

# STABILITY FIX: Move Spark Temp OUTSIDE project folder
TEMP_DIR = Path.home() / 'FinPulse_Spark_Temp'
os.makedirs(TEMP_DIR, exist_ok=True)

# JIT Stability Fix Reference
EXCLUDE_FILE = PROJECT_ROOT / 'exclude.txt'

# ── Environment ───────────────────────────────────────────────
if "SPARK_HOME" in os.environ and not Path(os.environ["SPARK_HOME"]).is_dir():
    del os.environ["SPARK_HOME"]

os.environ["JAVA_HOME"]             = r"C:\Program Files\Microsoft\jdk-11.0.30.7-hotspot"
os.environ["HADOOP_HOME"]           = str(PROJECT_ROOT / "hadoop")
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

for p in [
    Path(os.environ["JAVA_HOME"]) / "bin",
    Path(os.environ["HADOOP_HOME"]) / "bin",
]:
    if p.is_dir() and str(p) not in os.environ["PATH"]:
        os.environ["PATH"] = str(p) + os.pathsep + os.environ["PATH"]

# ── Paths ─────────────────────────────────────────────────────
BRONZE_STOCKS_PATH =str(PROJECT_ROOT / "data" / "bronze" / "stocks")
ICEBERG_WAREHOUSE = os.environ.get(
    "ICEBERG_WAREHOUSE",
    str(PROJECT_ROOT / "data" / "silver" / "iceberg_warehouse")
)
ICEBERG_JAR = str(PROJECT_ROOT / "jars" /
   "iceberg-spark-runtime-3.5_2.12-1.4.3.jar")

PIPELINE_VERSION    = "v1.0"
INGESTION_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"Project root     : {PROJECT_ROOT}")
print(f"Bronze stocks    : {BRONZE_STOCKS_PATH}")
print(f"Iceberg warehouse: {ICEBERG_WAREHOUSE}")
print(f"JAR exists       : {Path(ICEBERG_JAR).exists()}")

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType, BooleanType, IntegerType
from pyspark.sql.window import Window

# -- Spark Session with Stability Slack ------------------
spark_temp_path = str(TEMP_DIR).replace('\\', '/')
exclude_cfg_path = str(EXCLUDE_FILE).replace('\\', '/')
JVM_OPTS = (
    '-Dorg.apache.hadoop.io.nativeio.NativeIO.Windows.should_use_native_io=false'
    f' -Djava.io.tmpdir="{spark_temp_path}"'
    f' -XX:CompileCommandFile="{exclude_cfg_path}"'
    ' -XX:+UseG1GC'
)

spark = SparkSession.builder \
    .appName("FinPulse-Silver-Stocks") \
    .master("local[*]") \
    .config("spark.jars", ICEBERG_JAR) \
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local",
            "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", ICEBERG_WAREHOUSE) \
    .config("spark.driver.memory", "2500m") \
    .config("spark.executor.memory", "2500m") \
    .config("spark.driver.extraJavaOptions", JVM_OPTS) \
    .config("spark.executor.extraJavaOptions", JVM_OPTS) \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .config("spark.sql.iceberg.vectorization.enabled", "false") \
    .config("spark.hadoop.fs.file.impl",
            "org.apache.hadoop.fs.LocalFileSystem") \
    .config("spark.hadoop.fs.verify.checksum", "false") \
    .config("spark.network.timeout", "1200s") \
    .config("spark.executor.heartbeatInterval", "150s") \
    .config("spark.sql.defaultCatalog", "local") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"\n Spark + Iceberg ready | version: {spark.version}")


Project root     : C:\FinPulse Project
Bronze stocks    : C:\FinPulse Project\data\bronze\stocks
Iceberg warehouse: C:\FinPulse Project\data\silver\iceberg_warehouse
JAR exists       : True

 Spark + Iceberg ready | version: 3.5.8


In [5]:
def load_bronze_stocks(bronze_path: str):
    print("=" * 60)
    print("LOADING BRONZE STOCK DATA")
    print("=" * 60)

    p = Path(bronze_path)
    partitions = sorted(d for d in os.listdir(bronze_path)
                 if d.startswith("date="))
    latest = partitions[-1]
    
    # Robust Path Resolution
    partition_dir = p / latest
    file_path = partition_dir / "stocks_raw.csv"
    
    if not file_path.exists():
        csv_files = sorted([f for f in os.listdir(partition_dir) if f.endswith(".csv")])
        if not csv_files:
            raise FileNotFoundError(f"No CSV files found in partition: {partition_dir}")
        file_path = partition_dir / csv_files[-1]
        print(f"INFO: stocks_raw.csv not found. Picking latest CSV: {file_path.name}")

    file_path = str(file_path)
    print(f"Source: {file_path}")
    print(f"Partition: {latest}")                 

    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(file_path) 
    
    print(f"Records loaded : {df.count():,}")
    print(f"Columns        : {df.columns}")
    print(f"Tickers        : {[r[0] for r in df.select('ticker').distinct().collect()]}")

    return df 

bronze_stocks_df = load_bronze_stocks(BRONZE_STOCKS_PATH)
bronze_stocks_df.printSchema()


LOADING BRONZE STOCK DATA
INFO: stocks_raw.csv not found. Picking latest CSV: stocks_raw_185807.csv
Source: C:\FinPulse Project\data\bronze\stocks\date=2026-03-29\stocks_raw_185807.csv
Partition: date=2026-03-29
Records loaded : 2,505
Columns        : ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits', 'ticker', 'ingestion_timestamp', 'data_source']
Tickers        : ['AAPL', 'GOOGL', 'GS', 'MSFT', 'JPM']
root
 |-- Date: timestamp (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: integer (nullable = true)
 |-- Dividends: double (nullable = true)
 |-- Stock Splits: double (nullable = true)
 |-- ticker: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- data_source: string (nullable = true)



In [6]:
def apply_stocks_silver_transformations(df):
    """
    Applies all Silver layer transformations for stock data.
    Calculates financial metrics for market analysis.
    """
    print("=" * 60)
    print("APPLYING STOCKS SILVER TRANSFORMATIONS")
    print("=" * 60)

    # Step 1 — Clean column names to lowercase
    for col in df.columns:
        df = df.withColumnRenamed(col, col.lower().replace(" ", "_"))
    print(f"Step 1 — Columns standardised: {df.columns}")

    # Step 2 — Cast date column properly
    df = df.withColumn("date", F.to_date(F.col("date")))
    print("Step 2 — Date column cast to DateType")

    # Step 3 — Calculate daily return
    # (close - open) / open * 100
    df = df.withColumn(
        "daily_return",
        F.round(
            (F.col("close") - F.col("open")) / F.col("open") * 100,
            4
        )
    )
    print("Step 3 — daily_return calculated: (close-open)/open*100")

    # Step 4 — Calculate price change from previous day
    # Using window function — lag() looks at previous row per ticker
    window_spec = Window \
        .partitionBy("ticker") \
        .orderBy("date")

    df = df.withColumn(
        "price_change",
        F.round(
            F.col("close") - F.lag("close", 1).over(window_spec),
            4
        )
    )
    print("Step 4 — price_change calculated using lag() window function")

    # Step 5 — Add is_positive_day flag
    df = df.withColumn(
        "is_positive_day",
        F.when(F.col("close") > F.col("open"), True).otherwise(False)
    )
    positive_days = df.filter(F.col("is_positive_day")).count()
    total_days = df.count()
    print(f"Step 5 — is_positive_day: {positive_days:,}/{total_days:,} positive days ({positive_days/total_days*100:.1f}%)")

    # Step 6 — Calculate price range (high - low)
    df = df.withColumn(
        "price_range",
        F.round(F.col("high") - F.col("low"), 4)
    )
    print("Step 6 — price_range added: high-low spread per day")

    # Step 7 — Calculate 7 day rolling average close price
    window_7d = Window \
        .partitionBy("ticker") \
        .orderBy("date") \
        .rowsBetween(-6, 0)

    df = df.withColumn(
        "moving_avg_7d",
        F.round(F.avg("close").over(window_7d), 4)
    )
    print("Step 7 — 7 day moving average calculated")

    # Step 8 — Calculate 30 day rolling average
    window_30d = Window \
        .partitionBy("ticker") \
        .orderBy("date") \
        .rowsBetween(-29, 0)

    df = df.withColumn(
        "moving_avg_30d",
        F.round(F.avg("close").over(window_30d), 4)
    )
    print("Step 8 — 30 day moving average calculated")

    # Step 9 — Add pipeline metadata
    df = df.withColumn("silver_processed_at", F.lit(INGESTION_TIMESTAMP))
    df = df.withColumn("pipeline_version", F.lit(PIPELINE_VERSION))
    print("Step 9 — Pipeline metadata added")

    print(f"\n✅ Stocks Silver transformations complete")
    print(f"Final record count: {df.count():,}")
    print(f"Final columns: {len(df.columns)}")

    return df

silver_stocks_df = apply_stocks_silver_transformations(bronze_stocks_df)
silver_stocks_df.printSchema()


APPLYING STOCKS SILVER TRANSFORMATIONS
Step 1 — Columns standardised: ['date', 'open', 'high', 'low', 'close', 'volume', 'dividends', 'stock_splits', 'ticker', 'ingestion_timestamp', 'data_source']
Step 2 — Date column cast to DateType
Step 3 — daily_return calculated: (close-open)/open*100
Step 4 — price_change calculated using lag() window function
Step 5 — is_positive_day: 1,334/2,505 positive days (53.3%)
Step 6 — price_range added: high-low spread per day
Step 7 — 7 day moving average calculated
Step 8 — 30 day moving average calculated
Step 9 — Pipeline metadata added

✅ Stocks Silver transformations complete
Final record count: 2,505
Final columns: 19
root
 |-- date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: integer (nullable = true)
 |-- dividends: double (nullable = true)
 |-- stock_splits: double (nullable = true)
 |-- ticker: string (nulla

In [7]:
def validate_stocks_silver(df):
    """
    Validates Silver stocks data quality before writing.
    """
    print("=" * 60)
    print("STOCKS SILVER DATA VALIDATION")
    print("=" * 60)

    checks_passed = 0
    checks_failed = 0

    # Check 1 — No negative close prices
    negative_close = df.filter(F.col("close") <= 0).count()
    if negative_close == 0:
        print(" Check 1 PASSED — No negative close prices")
        checks_passed += 1
    else:
        print(f" Check 1 FAILED — {negative_close} negative close prices")
        checks_failed += 1

    # Check 2 — All 5 tickers present
    tickers = [r[0] for r in df.select("ticker").distinct().collect()]
    expected = {"AAPL", "GOOGL", "MSFT", "JPM", "GS"}
    if set(tickers) == expected:
        print(f" Check 2 PASSED — All 5 tickers present: {tickers}")
        checks_passed += 1
    else:
        missing = expected - set(tickers)
        print(f" Check 2 FAILED — Missing tickers: {missing}")
        checks_failed += 1

    # Check 3 — No null close prices
    null_close = df.filter(F.col("close").isNull()).count()
    if null_close == 0:
        print(" Check 3 PASSED — No null close prices")
        checks_passed += 1
    else:
        print(f" Check 3 FAILED — {null_close} null close prices")
        checks_failed += 1

    # Check 4 — Daily return range sanity check
    # Stock shouldn't move more than 50% in a day normally
    extreme_returns = df.filter(
        (F.col("daily_return") > 50) | (F.col("daily_return") < -50)
    ).count()
    if extreme_returns == 0:
        print(" Check 4 PASSED — No extreme daily returns (>50%)")
        checks_passed += 1
    else:
        print(f"  Check 4 INFO — {extreme_returns} extreme return days detected")
        checks_passed += 1

    # Check 5 — Stats per ticker
    print("\n── Ticker Summary ──")
    df.groupBy("ticker").agg(
        F.count("date").alias("trading_days"),
        F.round(F.avg("close"), 2).alias("avg_close"),
        F.round(F.avg("daily_return"), 4).alias("avg_daily_return_pct"),
        F.round(F.avg("price_range"), 2).alias("avg_daily_range")
    ).orderBy("ticker").show()

    print(f"\nValidation complete — {checks_passed} passed, {checks_failed} failed")

    if checks_failed > 0:
        raise Exception(f" {checks_failed} validation checks failed. Pipeline stopped.")

    return True

validate_stocks_silver(silver_stocks_df)

STOCKS SILVER DATA VALIDATION
 Check 1 PASSED — No negative close prices
 Check 2 PASSED — All 5 tickers present: ['AAPL', 'GOOGL', 'GS', 'MSFT', 'JPM']
 Check 3 PASSED — No null close prices
 Check 4 PASSED — No extreme daily returns (>50%)

── Ticker Summary ──
+------+------------+---------+--------------------+---------------+
|ticker|trading_days|avg_close|avg_daily_return_pct|avg_daily_range|
+------+------------+---------+--------------------+---------------+
|  AAPL|         501|   228.08|              0.0911|            4.7|
| GOOGL|         501|   206.94|              0.0573|           4.86|
|    GS|         501|   625.85|              0.1006|          14.81|
|   JPM|         501|   252.96|              0.0702|           4.98|
|  MSFT|         501|   440.79|             -0.0288|           7.74|
+------+------------+---------+--------------------+---------------+


Validation complete — 4 passed, 0 failed


True

In [8]:
def write_stocks_iceberg(df, iceberg_warehouse):
    """
    Writes Silver stocks as Apache Iceberg table.
    Same catalog as transactions — unified finpulse namespace.
    """
    print("=" * 60)
    print("WRITING STOCKS SILVER — Apache Iceberg")
    print("=" * 60)

    spark.sql("CREATE NAMESPACE IF NOT EXISTS local.finpulse")
    print(" Namespace local.finpulse ready")

    df.writeTo("local.finpulse.silver_stocks") \
      .tableProperty("format-version", "2") \
      .tableProperty("write.parquet.compression-codec", "snappy") \
      .createOrReplace()

    print(" Iceberg table written: local.finpulse.silver_stocks")

    count = spark.table("local.finpulse.silver_stocks").count()
    print(f" Records in Iceberg table: {count:,}")

    print("\n── Iceberg Snapshots ──")
    spark.sql("""
        SELECT snapshot_id, committed_at, operation
        FROM local.finpulse.silver_stocks.snapshots
    """).show(truncate=False)

    print("── Table Schema ──")
    spark.sql("""
        DESCRIBE TABLE local.finpulse.silver_stocks
    """).show(truncate=False)

    return "local.finpulse.silver_stocks"

output_stocks = write_stocks_iceberg(silver_stocks_df, ICEBERG_WAREHOUSE)

WRITING STOCKS SILVER — Apache Iceberg
 Namespace local.finpulse ready
 Iceberg table written: local.finpulse.silver_stocks
 Records in Iceberg table: 2,505

── Iceberg Snapshots ──
+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|4695129907907832584|2026-04-10 00:13:23.551|overwrite|
+-------------------+-----------------------+---------+

── Table Schema ──
+-------------------+---------+-------+
|col_name           |data_type|comment|
+-------------------+---------+-------+
|date               |date     |NULL   |
|open               |double   |NULL   |
|high               |double   |NULL   |
|low                |double   |NULL   |
|close              |double   |NULL   |
|volume             |int      |NULL   |
|dividends          |double   |NULL   |
|stock_splits       |double   |NULL   |
|ticker             |string   |NULL   |
|ingestion_timestamp|timestamp|NULL   

In [ ]:
# ============================================================
# STANDARDIZATION: Export Silver Stocks to Parquet
# Purpose: Align project structure with Transactions (parquet-based)
# ============================================================

SILVER_STOCKS_PARQUET = PROJECT_ROOT / "data" / "silver" / "stocks"
os.makedirs(SILVER_STOCKS_PARQUET, exist_ok=True)

# Use the same date partition as Bronze
silver_output_path = SILVER_STOCKS_PARQUET / latest

print(f"Exporting Silver Stocks to: {silver_output_path}")
silver_stocks_df.write.mode("overwrite").parquet(str(silver_output_path))
print("DONE: Stocks Silver Parquet export complete")


In [ ]:
# ============================================================
# STANDARDIZATION: Export Silver Stocks to Parquet
# Purpose: Align project structure with Transactions (parquet-based)
# ============================================================

SILVER_STOCKS_PARQUET = PROJECT_ROOT / "data" / "silver" / "stocks"
os.makedirs(SILVER_STOCKS_PARQUET, exist_ok=True)

# Use the same date partition as Bronze
silver_output_path = SILVER_STOCKS_PARQUET / latest

print(f"Exporting Silver Stocks to: {silver_output_path}")
silver_stocks_df.write.mode("overwrite").parquet(str(silver_output_path))
print("✅ Stocks Silver Parquet export complete")


In [9]:
print("=" * 60)
print("FINPULSE SILVER STOCKS — COMPLETE SUMMARY")
print("=" * 60)
print(f"Source          : Bronze stocks (Yahoo Finance)")
print(f"Tickers         : AAPL, GOOGL, MSFT, JPM, GS")
print(f"Records         : {silver_stocks_df.count():,}")
print(f"Columns         : {len(silver_stocks_df.columns)}")
print(f"New columns added:")
print(f"  - daily_return      (% gain/loss per day)")
print(f"  - price_change      (absolute $ change from yesterday)")
print(f"  - is_positive_day   (True if close > open)")
print(f"  - price_range       (high-low spread)")
print(f"  - moving_avg_7d     (7 day rolling average)")
print(f"  - moving_avg_30d    (30 day rolling average)")
print(f"  - silver_processed_at")
print(f"  - pipeline_version")
print(f"Iceberg table   : local.finpulse.silver_stocks")
print(f"Storage format  : Apache Iceberg format-version 2")
print("=" * 60)
print(" SILVER STOCKS COMPLETE")
print("=" * 60)

spark.stop()

FINPULSE SILVER STOCKS — COMPLETE SUMMARY
Source          : Bronze stocks (Yahoo Finance)
Tickers         : AAPL, GOOGL, MSFT, JPM, GS
Records         : 2,505
Columns         : 19
New columns added:
  - daily_return      (% gain/loss per day)
  - price_change      (absolute $ change from yesterday)
  - is_positive_day   (True if close > open)
  - price_range       (high-low spread)
  - moving_avg_7d     (7 day rolling average)
  - moving_avg_30d    (30 day rolling average)
  - silver_processed_at
  - pipeline_version
Iceberg table   : local.finpulse.silver_stocks
Storage format  : Apache Iceberg format-version 2
 SILVER STOCKS COMPLETE


In [10]:
import shutil
from pathlib import Path
import os

# Environment-aware drive path
GDRIVE_ROOT = Path(os.environ.get('GDRIVE_FINPULSE_ROOT', 'G:/My Drive/FinPulse'))
LOCAL_ICEBERG = Path(r"str(PROJECT_ROOT)\data\silver\iceberg_warehouse")
DRIVE_ICEBERG = GDRIVE_ROOT / "data" / "silver" / "iceberg_warehouse"

print("=" * 60)
print("SYNCING ICEBERG WAREHOUSE TO GOOGLE DRIVE")
print("=" * 60)
print(f"Target: {DRIVE_ICEBERG}")

if LOCAL_ICEBERG.exists():
    try:
        if DRIVE_ICEBERG.exists():
            shutil.rmtree(DRIVE_ICEBERG)
            print(" Removed old Drive copy")

        shutil.copytree(LOCAL_ICEBERG, DRIVE_ICEBERG)

        total_size = sum(f.stat().st_size for f in LOCAL_ICEBERG.rglob('*') if f.is_file())
        print(f" Full Iceberg warehouse synced to Drive")
        print(f"   Contains: silver_transactions + silver_stocks tables")
        print(f"   Total size: {total_size / (1024*1024):.1f} MB")
        print(f"\n Google Drive backup complete")
    except Exception as e:
        print(f" Sync failed: {e}")
        print(" (Ensure Google Drive is mounted and GDRIVE_FINPULSE_ROOT is correct)")
else:
    print(" Local Iceberg warehouse not found")


SYNCING ICEBERG WAREHOUSE TO GOOGLE DRIVE
Target: G:\My Drive\FinPulse\data\silver\iceberg_warehouse
 Removed old Drive copy
 Full Iceberg warehouse synced to Drive
   Contains: silver_transactions + silver_stocks tables
   Total size: 926.1 MB

 Google Drive backup complete
